### [MLJAR](https://github.com/mljar/mljar-supervised?tab=readme-ov-file)

In [1]:
!pip install --no-cache-dir mljar-supervised

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from supervised.automl import AutoML # mljar-supervised

/home/martin/miniconda3/envs/mljar/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

from sklearn.model_selection import train_test_split
train = pd.read_csv('../data/train.csv', sep=",")
test = pd.read_csv('../data/test.csv')

import sklearn
y = train['SalePrice']
X = train.drop('SalePrice', axis=1)
categorical = [var for var in X.columns if X[var].dtype=='O']
numerical = [var for var in X.columns if X[var].dtype!='O']
X[categorical] = X[categorical].fillna('None')

# auto-sklearn can not deal with categorical variables
X= pd.concat([pd.get_dummies(X[categorical], dummy_na=True), X[numerical]], axis=1)
# enc = OneHotEncoder(handle_unknown='ignore')
# X = pd.concat([pd.DataFrame(enc.fit_transform(X[categorical])), X[numerical]], axis=1)

y = np.log1p(y)
to_convert = {i: int for i,j in list(zip(X.dtypes.index, X.dtypes.values)) if j == 'bool'}
X = X.astype(to_convert)
X_train, X_test, y_train, y_test = \
    train_test_split(X, y, random_state=42, test_size=0.2)

In [3]:


# train models with AutoML
automl = AutoML(mode="Explain")
automl.fit(X_train, y_train)

# compute the MSE on test data
predictions = automl.predict(X_test)
print("Test MSE:", np.sqrt(mean_squared_error(y_test, predictions)))

AutoML directory: AutoML_4
The task is regression with evaluation metric rmse
AutoML will use algorithms: ['Baseline', 'Linear', 'Decision Tree', 'Random Forest', 'Xgboost', 'Neural Network']
AutoML will ensemble available models
AutoML steps: ['simple_algorithms', 'default_algorithms', 'ensemble']
* Step simple_algorithms will try to check up to 3 models
1_Baseline rmse 0.375655 trained in 0.16 seconds
2_DecisionTree rmse 0.226412 trained in 9.64 seconds
3_Linear rmse 0.167113 trained in 3.75 seconds
* Step default_algorithms will try to check up to 3 models
4_Default_Xgboost rmse 0.120281 trained in 7.33 seconds
5_Default_NeuralNetwork rmse 0.154929 trained in 7.46 seconds
6_Default_RandomForest rmse 0.167743 trained in 12.21 seconds
* Step ensemble will try to check up to 1 model
Ensemble rmse 0.112552 trained in 0.08 seconds
AutoML fit time: 44.51 seconds
AutoML best model: Ensemble
Test MSE: 0.13441719928513732
